# 分析目標
Peter Lynch 在 《彼得林區：選股戰略》裡提出星期一股市比較容易下跌的趣聞，本篇分析的目的即為驗證此趣聞是否為適用於今日 NASDAQ 100，不構成任何投資建議。

方法：以 yfinance 抓取 2000 年來所有台股日收盤價，以兩種分母檢視各 weekday 的漲跌比例，最後以卡方分配檢視是否有顯著差異。
1. P(down∣weekday) 相對機率 -> 所有禮拜一的 K 線裡，有多少是綠色？
2. P(weekday|down) 貢獻比例 -> 所有的綠 K 裡，哪一天比例最高？

兩個方法的差別在於：有幾個蛋糕？
相對機率：有五個蛋糕，裡面的奶油有兩種顏色：紅跟綠，每一個蛋糕的顏色比例稍微有那麼一點不同，第一個蛋糕的紅色多一點點，第二個綠色多一點點，以此類推。
貢獻比例：只有一個蛋糕，裡面的奶油有五種顏色：紅橙黃綠藍，分別代表一到五。

# 結論
以 NASDAQ 100 二十年收盤數據顯示，無論以相對機率或者超額貢獻來看，禮拜一的下跌比例都不是最高的。雖能觀察到周間有些微的比例差異，但不具備顯著性。若非得選出最容易下跌的天，那也得選**禮拜五**而不是禮拜一。值得注意的是，所謂的「容易」，也僅僅是微乎其微的差異而已。 \
總的來說，彼得林區的觀察在今日 NASDQA 指數上並不成立。若要依此研究進行操作，請自負後果～

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf

from plotnine import (
    ggplot, aes, geom_col, geom_boxplot, geom_density, facet_wrap,
    labs, theme, element_text, scale_x_discrete
)

from scipy.stats import chi2_contingency

In [12]:
# ==============
# 抓 2000-2026 資料（Yahoo Finance via yfinance）
# ==============
SYMBOL = "^NDX"          # 台股加權指數
START  = "2000-01-01"
END    = "2026-12-31"

df = yf.download(SYMBOL, start=START, end=END, auto_adjust=False, progress=False)

if df.empty:
    raise RuntimeError("抓不到資料。可能是網路、SYMBOL、或 Yahoo Finance 暫時抽風。")

# 確保欄位與 index
df = df.reset_index()  # 變成 Date 欄
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date")
df.columns = df.columns.get_level_values(0)

### 相對機率

In [4]:
# ==============
#    用前一天漲跌值，計算星期一～星期五每天的漲跌次數
#    這裡用「點數變動」：Close - 前一日 Close
#    也順便算報酬率，方便看分佈（可二選一）
# ==============
df["prev_close"] = df["Close"].shift(1)
df["delta"] = df["Close"] - df["prev_close"]
df["ret"] = df["Close"].pct_change()

# 去掉第一天（沒有前一天）
df = df.dropna(subset=["delta", "ret"]).copy()

# 週幾：Monday=0 ... Sunday=6
df["weekday"] = df["Date"].dt.dayofweek

# 只保留週一～週五（台股本來就多數是這樣，但保險）
df = df[df["weekday"].between(0, 4)].copy()

weekday_map = {0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri"}
df["weekday_name"] = df["weekday"].map(weekday_map)

order = ["Mon", "Tue", "Wed", "Thu", "Fri"]
df["weekday_name"] = pd.Categorical(df["weekday_name"], categories=order, ordered=True)

# 漲跌標記（把 delta == 0 當作 flat）
df["move"] = np.where(df["delta"] > 0, "up",
               np.where(df["delta"] < 0, "down", "flat"))

# 統計：每個 weekday 的 up/down/flat 次數與比例
counts = (
    df.groupby(["weekday_name", "move"])
      .size()
      .reset_index(name="n")
)

totals = df.groupby("weekday_name").size().reset_index(name="total")
counts = counts.merge(totals, on="weekday_name", how="left")
counts["pct"] = counts["n"] / counts["total"]

# 只看「跌」次數
down_counts = counts[counts["move"] == "down"].copy()

print("=== Up/Down/Flat counts by weekday ===")
print(counts.pivot(index="weekday_name", columns="move", values="n").fillna(0).astype(int))
print("\n=== Down ratio by weekday ===")
print(down_counts[["weekday_name", "n", "total", "pct"]].sort_values("weekday_name"))

=== Up/Down/Flat counts by weekday ===
move          down  flat   up
weekday_name                 
Mon            542     3  696
Tue            623     0  736
Wed            596     0  762
Thu            607     1  724
Fri            636     0  692

=== Down ratio by weekday ===
   weekday_name    n  total       pct
0           Mon  542   1241  0.436745
3           Tue  623   1359  0.458425
6           Wed  596   1358  0.438881
9           Thu  607   1332  0.455706
12          Fri  636   1328  0.478916


/tmp/ipykernel_3763077/456006267.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
/tmp/ipykernel_3763077/456006267.py:36: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


In [48]:
# 只分 Down / Not Down
df["is_down"] = (df["move"] == "down").astype(int)

table = pd.crosstab(df["weekday_name"], df["is_down"])
table.columns = ["Not_down", "Down"]



chi2, p_value, dof, expected = chi2_contingency(table.loc[["Mon", "Fri"], :])
print(table.loc[["Mon", "Fri"], :])
print("chi2 =", chi2)
print("p-value =", p_value)

              Not_down  Down
weekday_name                
Mon                689   539
Fri                685   630
chi2 = 3.9637828770352495
p-value = 0.046489118872263764


### 超額貢獻 (lift value)

In [11]:
# 先拿下跌日
down_df = df[df["move"] == "down"].copy()

# 每個 weekday 的下跌天數
down_weekday = (
    down_df.groupby("weekday_name")
           .size()
           .reset_index(name="down_n")
)

# 分母：所有下跌天數
down_total = len(down_df)
down_weekday["pct_of_all_down_days"] = down_weekday["down_n"] / down_total

# 計算隨機比例與超額貢獻
# 每個 weekday 佔所有交易日的比例
weekday_freq = (
    df.groupby("weekday_name")
      .size()
      .reset_index(name="n_days")
)
weekday_freq["expected_share"] = weekday_freq["n_days"] / weekday_freq["n_days"].sum()
compare = down_weekday.merge(
    weekday_freq[["weekday_name", "expected_share"]],
    on="weekday_name"
)
compare["lift"] = compare["pct_of_all_down_days"] - compare["expected_share"]



print("=== Distribution of ALL down days across weekdays ===")
print(down_weekday.sort_values("weekday_name"))
print(f"Total down days = {down_total}")

print("\n")
print("=== Lift score of weekdays ===")
print(compare.sort_values("weekday_name"))

=== Distribution of ALL down days across weekdays ===
  weekday_name  down_n  pct_of_all_down_days
0          Mon     542              0.180426
1          Tue     623              0.207390
2          Wed     596              0.198402
3          Thu     607              0.202064
4          Fri     636              0.211718
Total down days = 3004


=== Lift score of weekdays ===
  weekday_name  down_n  pct_of_all_down_days  expected_share      lift
0          Mon     542              0.180426        0.187519 -0.007093
1          Tue     623              0.207390        0.205349  0.002041
2          Wed     596              0.198402        0.205198 -0.006796
3          Thu     607              0.202064        0.201269  0.000795
4          Fri     636              0.211718        0.200665  0.011053


/tmp/ipykernel_3763077/1735968361.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
/tmp/ipykernel_3763077/1735968361.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
